# Predicción de efectos secundarios de medicamentos

Sistema **PIMES** (*Predictor de Interacciones de Medicamentos y Efectos Secundarios*). Recibe la estructura química de un medicamento codificada como un vector binario de 881 subestructuras y devuelve, para cada uno de los 1385 efectos secundarios posibles, la probabilidad de que el medicamento lo cause.

Se trata de un problema de **clasificación multietiqueta** (un medicamento puede provocar varios efectos a la vez), por lo que se entrena una red neuronal *poco profunda* (MLP) con activación sigmoide en la salida y pérdida logística por etiqueta. El código de línea de comandos correspondiente se encuentra en `PIMES.py`; este cuaderno reproduce el mismo flujo de forma interactiva.

## 1. Importar librerías

In [ ]:
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)

## 2. Cargar los datos

Se leen dos archivos tabulares:

- `medicamentos.txt`: matriz de 888 medicamentos × 881 subestructuras químicas (1/0).
- `efectosSecundariosDeLosMedicamentos.txt`: matriz de 888 medicamentos × 1385 efectos secundarios (1/0).

Ambos archivos tienen la peculiaridad de que el encabezado contiene sólo los nombres de las columnas (881 o 1385) mientras que cada fila empieza con el nombre del medicamento. `pandas` detecta automáticamente ese desfase y coloca el nombre del medicamento como índice.

In [ ]:
DATA_DIR = os.path.join('..', 'datasets', 'datasets')

df_med = pd.read_csv(os.path.join(DATA_DIR, 'medicamentos.txt'), sep='\t', header=0, dtype=str)
df_ef  = pd.read_csv(os.path.join(DATA_DIR, 'efectosSecundariosDeLosMedicamentos.txt'), sep='\t', header=0, dtype=str)

subestructuras = list(df_med.columns)
efectos = list(df_ef.columns)
nombres_med = df_med.index.astype(str).tolist()
nombres_ef  = df_ef.index.astype(str).tolist()

X = df_med.to_numpy(dtype=np.float32)
Y = df_ef.to_numpy(dtype=np.int8)

assert nombres_med == nombres_ef, 'Los archivos deben tener los mismos medicamentos en el mismo orden.'

print(f'Medicamentos: {X.shape[0]}')
print(f'Subestructuras (entradas): {X.shape[1]}')
print(f'Efectos secundarios (salidas): {Y.shape[1]}')
print(f'Densidad de etiquetas positivas: {Y.mean():.4f}')

## 3. Construir el modelo

Red neuronal *poco profunda* con tres capas ocultas (512 → 256 → 128 neuronas, activación ReLU) y una capa de salida de 1385 unidades. Como el problema es multietiqueta, `MLPClassifier` aplica activación logística (sigmoide) independiente por neurona de salida y pérdida log-loss binaria por etiqueta. El optimizador es Adam con tasa de aprendizaje $10^{-3}$ y mini-lotes de 32 muestras.

In [ ]:
HIDDEN_LAYERS = (512, 256, 128)
MAX_ITER = 80
RANDOM_STATE = 42

def construir_modelo():
    return MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation='relu',
        solver='adam',
        learning_rate_init=1e-3,
        batch_size=32,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
        verbose=False,
    )

construir_modelo()

## 4. Validación cruzada 5-fold

Se evalúa la red con la técnica *5-fold cross-validation*: el conjunto se divide en cinco particiones; en cada iteración se entrena con cuatro y se evalúa con la restante. Como métrica global se usa el **AUC micro-promedio**, que aplana las matrices de etiquetas y puntajes en un único vector binario antes de calcular la curva ROC; es la métrica habitual cuando se quiere resumir el desempeño sobre 1385 etiquetas a la vez.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
aucs = []
y_true_all, y_score_all = [], []

for fold, (idx_tr, idx_te) in enumerate(kf.split(X), 1):
    t0 = time.time()
    modelo = construir_modelo()
    modelo.fit(X[idx_tr], Y[idx_tr])
    y_score = modelo.predict_proba(X[idx_te])
    y_true = Y[idx_te]
    fpr, tpr, _ = roc_curve(y_true.ravel(), y_score.ravel())
    a = auc(fpr, tpr)
    aucs.append(a)
    y_true_all.append(y_true)
    y_score_all.append(y_score)
    print(f'Fold {fold}: AUC micro = {a:.4f}  ({time.time()-t0:.1f}s)')

y_true_cat = np.concatenate(y_true_all)
y_score_cat = np.concatenate(y_score_all)
fpr_g, tpr_g, _ = roc_curve(y_true_cat.ravel(), y_score_cat.ravel())
auc_global = auc(fpr_g, tpr_g)
print()
print(f'AUC promedio: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'AUC global  : {auc_global:.4f}')

## 5. Curva ROC

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(fpr_g, tpr_g, label=f'micro-promedio (AUC = {auc_global:.4f})', color='C0')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='azar')
plt.xlim(0, 1); plt.ylim(0, 1.05)
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC — PIMES (5-fold CV)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Entrenar el modelo final y guardarlo

Una vez validado el desempeño, se entrena un modelo final sobre la totalidad de los 888 medicamentos y se persiste en disco junto con la lista de subestructuras y efectos. Ese mismo archivo `modelo_PIMES.joblib` es el que carga `PIMES.py -p` para hacer predicciones.

In [ ]:
modelo_final = construir_modelo()
modelo_final.fit(X, Y)

bundle = {
    'modelo': modelo_final,
    'subestructuras': subestructuras,
    'efectos': efectos,
    'auc_promedio': float(np.mean(aucs)),
    'auc_global': float(auc_global),
    'hidden_layers': HIDDEN_LAYERS,
}
joblib.dump(bundle, 'modelo_PIMES.joblib')
print('Modelo guardado en modelo_PIMES.joblib')

## 7. Predicción para medicamentos sin efectos conocidos

Se carga el modelo guardado y se predicen los efectos secundarios para los medicamentos del archivo `medicamentosSinEfectosSecun.txt`. Para cada medicamento se muestran los efectos en orden descendente de probabilidad (top 10).

In [ ]:
bundle = joblib.load('modelo_PIMES.joblib')
modelo = bundle['modelo']
efectos_entrenamiento = bundle['efectos']

df_sin = pd.read_csv(os.path.join(DATA_DIR, 'medicamentosSinEfectosSecun.txt'), sep='\t', header=0, dtype=str)
X_sin = df_sin.to_numpy(dtype=np.float32)
ids_sin = df_sin.index.astype(str).tolist()

with open(os.path.join(DATA_DIR, 'listaDeEfectosSecun.txt'), encoding='utf-8') as f:
    efectos_consulta = [linea.strip() for linea in f if linea.strip()]

indice = {e: i for i, e in enumerate(efectos_entrenamiento)}
columnas = [indice[e] for e in efectos_consulta]

proba = modelo.predict_proba(X_sin)
proba_consulta = proba[:, columnas]

for pubmed_id, fila in list(zip(ids_sin, proba_consulta))[:3]:
    orden = np.argsort(-fila)[:10]
    print(f'\nPubMed ID {pubmed_id} — top 10 efectos más probables:')
    for j in orden:
        print(f'  {efectos_consulta[j]:<40s}  {fila[j]:.4f}')

## Resumen

Se entrenó una red neuronal poco profunda (MLP con 3 capas ocultas, salidas sigmoide y pérdida log-loss binaria por etiqueta) sobre las matrices de subestructuras químicas y efectos secundarios de 888 medicamentos. Con *5-fold cross-validation* se obtuvo un AUC micro-promedio competitivo, y el modelo final puede usarse para predecir los efectos más probables de medicamentos que no fueron vistos durante el entrenamiento. El sistema completo se entrega como el script de línea de comandos `PIMES.py` con los modos `-e` (entrenamiento) y `-p` (predicción).